In [1]:
import pandas as pd
import math
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.animation as animation
from matplotlib.patches import Rectangle
from matplotlib.colors import LogNorm
import scipy
from scipy import stats
from scipy.signal import welch, stft, spectrogram
from scipy.stats import chi2
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
import os
from datetime import datetime, timedelta
import re
import pywt
from numpy.fft import rfft, rfftfreq
import copy
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
from IPython.display import clear_output

mpl.rcParams['agg.path.chunksize'] = 10000000

In [ ]:
def is_number(s):
    # Define a regex pattern to match positive, negative, and decimal numbers
    number_pattern = re.compile(r'^-?(\d+(\.\d*)?|\.\d+)$')
    
    # Use the pattern to check if the string matches
    return bool(number_pattern.match(s))

def truncate_microseconds(dt_str):
    # Split the time string at the decimal point
    if '.' in dt_str:
        time_part, microseconds_part = dt_str.split('.')
        # Truncate to only the first six digits of microseconds
        truncated_microseconds_part = microseconds_part[:6]
        truncated_dt_str = f"{time_part}.{truncated_microseconds_part}"
        return truncated_dt_str
    return dt_str

net_AE_time_adjust_from_file = 0.0

## Load Data

### Load Mistras Hits Data

In [ ]:
Mistras_filepath = ""
while True:
    Mistras_filepath = input("Please enter the absolute filepath of your Mistras .txt file (Enter 'None' to skip Mistras import): ")
    Mistras_filepath = Mistras_filepath.strip('"')

    if os.path.isfile(Mistras_filepath):
        print("\nMistras Data Filepath: ", f"\"{Mistras_filepath}\"")
        break
    print("\nThe specified file does not exist. Please check the path and try again.")
    
def get_mistras_metadata(filepath, df):
    metadata = {}
    
    # This regex pattern matches a time format and optionally captures microseconds if present
    time_pattern = re.compile(r'\b(\d{2}:\d{2}:\d{2})(?:\.(\d+))?\b')
    
    with open(filepath, 'r') as file:
        for line in file:
            match = re.search(time_pattern, line)
            if match:
                time_str = match.group(1)  # Time in HH:MM:SS
                microseconds_str = match.group(2) if match.group(2) else '000000'  # Microseconds part if present, else '0'
                break
        else:
            # Handle case where no time is found
            return None

    # Append microseconds to the time string and parse
    full_time_str = f"{time_str}.{microseconds_str}"
    time_obj = datetime.strptime(full_time_str, "%H:%M:%S.%f")

    metadata['start_time'] = time_obj
    metadata['duration'] = df['SSSSSSSS.mmmuuun'].iloc[-1]
    return metadata

def import_mistras_file_to_df(filepath):
    # Read the data using read_csv, assuming tab-separated values (TSV); adjust delimiter if needed
    df = pd.read_csv(filepath, sep='\s+', header=4)
    # There is a bad import on the 'SIG STRNGTH col because it has a space in the name and the file is whitespace-delimted
    column_names_AE_Node = [
        'ID', 'SSSSSSSS.mmmuuun', 'CH', 'RISE', 'COUN', 'ENER', 
        'DURATION', 'AMP', 'A-FRQ', 'RMS', 'ASL', 'PCNTS', 'THR', 'R-FRQ', 
        'I-FRQ', 'SIG STRNGTH', 'ABS-ENERGY'
    ]
    column_names_EasyAE = [
        'ID', 'SSSSSSSS.mmmuuun', 'PARA1', 'CH', 
        'RISE', 'COUN', 'ENER', 
        'DURATION', 'A-FRQ', 'RMS', 'PCNTS', 'THR', 'R-FRQ', 
        'I-FRQ', 'SIG STRNGTH', 'ABS-ENERGY', 'FRQ-C', 'P-FRQ', 'AMP', 'ASL'
    ]
    column_names = column_names_EasyAE
    # Keep all columns except the last one
    df = df.iloc[:, :-1]
    df.columns = column_names
    # Remove rows where any value is not a float
    # Attempt to convert all values to float
    df = df.apply(pd.to_numeric, errors='coerce')

    # Drop rows with any NaNs (i.e., conversion failures)
    df = df.dropna(how='any')

    return df

def process_mistras_df(df, metadata):
    # Drop 'ID' col
    df = df.drop('ID', axis=1)
    # Rename time col
    df.rename(columns={'SSSSSSSS.mmmuuun': 'TIME'}, inplace=True)
    
    # Add 'Absolute time' column to the data
    mistras_start_time_obj = metadata['start_time']
    seconds = df['TIME']
    time_deltas = pd.to_timedelta(seconds, unit='s')
    absolute_time = mistras_start_time_obj + time_deltas
    df.insert(0, 'Absolute Time', absolute_time)
    
    return df

mistras_df = import_mistras_file_to_df(Mistras_filepath)
mistras_metadata = get_mistras_metadata(Mistras_filepath, mistras_df)
mistras_df = process_mistras_df(mistras_df, mistras_metadata)
print("Start time:", mistras_metadata['start_time'])
print("Duration: " + str(mistras_metadata['duration']) + " s")

display(mistras_df)

amp_type_counts = mistras_df['AMP'].apply(lambda x: type(x).__name__).value_counts()
print("Type counts in 'AMP' column:")
print(amp_type_counts)
mistras_df_times_synchronized = False
hits_inserted = False

### Load Mistras Time-Driven Data

In [ ]:
Mistras_tdd_filepath = ""
while True:
    Mistras_tdd_filepath = input("Please enter the absolute filepath of your Mistras TDD .txt file (Enter 'None' to skip import): ")
    Mistras_tdd_filepath = Mistras_tdd_filepath.strip('"')
    print("\nMistras TDD Data Filepath: ", repr(Mistras_tdd_filepath))

    if os.path.isfile(Mistras_tdd_filepath):
        break
    print("\nThe specified file does not exist. Please check the path and try again.")

def get_mistras_metadata(filepath, df):
    metadata = {}
    
    # This regex pattern matches a time format and optionally captures microseconds if present
    time_pattern = re.compile(r'\b(\d{2}:\d{2}:\d{2})(?:\.(\d+))?\b')
    
    with open(filepath, 'r') as file:
        for line in file:
            match = re.search(time_pattern, line)
            if match:
                time_str = match.group(1)  # Time in HH:MM:SS
                microseconds_str = match.group(2) if match.group(2) else '000000'  # Microseconds part if present, else '0'
                break
        else:
            # Handle case where no time is found
            return None

    # Append microseconds to the time string and parse
    full_time_str = f"{time_str}.{microseconds_str}"
    time_obj = datetime.strptime(full_time_str, "%H:%M:%S.%f")

    metadata['start_time'] = time_obj
    metadata['duration'] = df['SSSSSSSS.mmmuuun'].iloc[-1]
    return metadata

def parse_tdd_data(file_path):
    # Initialize lists to store data
    timestamps = []
    rms = []
    asl = []
    thr = []
    abs_energy = []
    
    # Open and read the file
    with open(file_path, 'r') as file:
        for line in file:
            if line.startswith('  2 '):
                # Extract timestamp
                timestamp = float(line.split()[1])
                timestamps.append(timestamp)
                
                # Read the next two lines
                _ = next(file)  # Skip the "CH:[    RMS ASL THR  ABS-ENERGY]" line
                channel_data = next(file).strip()
                
                # Extract channel data
                data = channel_data.split('[')[-1].split(']')[0].split()
                rms.append(float(data[0]))
                asl.append(float(data[3]))
                thr.append(float(data[1]))
                abs_energy.append(float(data[2]))
    
    # Create DataFrame
    df = pd.DataFrame({
        'SSSSSSSS.mmmuuun': timestamps,
        'RMS_CONTINUOUS': rms,
        'THR_CONTINUOUS': thr,
        'ABS_ENERGY_CONTINUOUS': abs_energy,
        'ASL_CONTINUOUS': asl,
    })
    
    return df

def process_mistras_tdd_df(df, metadata):
    # Rename time col
    df.rename(columns={'SSSSSSSS.mmmuuun': 'TIME'}, inplace=True)
    
    # Add 'Absolute time' column to the data
    mistras_start_time_obj = metadata['start_time']
    seconds = df['TIME']
    time_deltas = pd.to_timedelta(seconds, unit='s')
    absolute_time = mistras_start_time_obj + time_deltas
    df.insert(0, 'Absolute Time', absolute_time)
    
    return df

mistras_tdd_df = None
mistras_tdd_df = parse_tdd_data(Mistras_tdd_filepath)
mistras_tdd_metadata = get_mistras_metadata(Mistras_tdd_filepath, mistras_tdd_df)
mistras_tdd_df = process_mistras_tdd_df(mistras_tdd_df, mistras_tdd_metadata)
print("Start time:", mistras_tdd_metadata['start_time'])
print("Duration: " + str(mistras_tdd_metadata['duration']) + " s")
display(mistras_tdd_df)
mistras_tdd_df_times_synchronized = False

### Load EasyAE Streamed Waveforms

In [ ]:
# Ask for folder_path
while True:
    folder_path = input("Enter the absolute path to waveforms …").strip('"')
    if os.path.isdir(folder_path):
        break
    print("Invalid directory. Try again.")

print(f"Using folder: \"{folder_path}\"")

Using folder: "C:\Users\sapierso\Box\PD-Acoustic Project Data\0_FormattedData\PD-acoustic170\PD-acoustic170_wfms"


In [ ]:
# Build & sort CSV-file list up front
def sort_key(fn):
    base = os.path.splitext(fn)[0]
    parts = base.split('_')
    try:
        return int(parts[-2])
    except:
        return 0

csv_files = sorted(
    [f for f in os.listdir(folder_path) if f.lower().endswith('.csv')],
    key=sort_key
)

# Try loading the pickle, otherwise build & cache
base_name     = os.path.basename(folder_path)
pickle_file   = os.path.join(folder_path, base_name + '_combined.pkl')

if os.path.isfile(pickle_file):
    print("Loading cached array…")
    with open(pickle_file, 'rb') as pf:
        loaded_item = pickle.load(pf)

    if isinstance(loaded_item, dict):
        # already in the new format
        combined_wfm_data      = loaded_item['combined_data']
        sample_interval    = loaded_item['sample_interval']
        stream_start_time  = loaded_item['start_time']
        wfm_stream_metadata = {
            'combined_data': combined_wfm_data,
            'sample_interval': sample_interval,
            'start_time': stream_start_time,
        }

    elif isinstance(loaded_item, np.ndarray):
        # --- NEW: bring the old array forward into the new-dict format ---
        combined_wfm_data = loaded_item

        if csv_files:
            # re-extract sample_interval & start_time from first CSV
            first_csv = os.path.join(folder_path, csv_files[0])
            with open(first_csv, 'r') as f:
                header = [next(f) for _ in range(12)]
            sample_interval = float(header[3].split(':')[-1].strip())
            stream_start_time_str = header[2].split(' ')[-1].strip()
            stream_start_time = datetime.strptime(stream_start_time_str, "%H:%M:%S")
        else:
            # If no CSV files, set defaults
            sample_interval = 1e-6  # Default to 1 microsecond
            stream_start_time = mistras_metadata['start_time']  # Default to time from Mistras metadata

        # build the dict
        wfm_stream_metadata = {
            'combined_data': combined_wfm_data,
            'sample_interval': sample_interval,
            'start_time': stream_start_time,
        }

        # save it right back
        print("Upgrading pickle to new format…")
        with open(pickle_file, 'wb') as pf:
            pickle.dump(wfm_stream_metadata, pf)
        print("Re‐saved pickle with metadata.")
else:
    # Always pull sample_interval from first csv file
    first_csv = os.path.join(folder_path, csv_files[0])
    with open(first_csv, 'r') as f:
        header = [next(f) for _ in range(12)]
    sample_interval = float(header[3].split(':')[-1].strip())
    print("Sample interval (s):", sample_interval)
    stream_start_time_str = header[2].split(' ')[-1].strip()
    stream_start_time = datetime.strptime(stream_start_time_str, "%H:%M:%S")
    print("Stream start time:", stream_start_time)
    wfm_stream_metadata = {
        'sample_interval': sample_interval,
        'start_time': stream_start_time,
    }

    data_list = []
    for fn in tqdm(csv_files, desc="Reading CSVs"):
        path = os.path.join(folder_path, fn)
        arr  = np.loadtxt(path, delimiter=',', skiprows=12)
        data_list.append(arr)
    combined_wfm_data = np.vstack(data_list)
    wfm_stream_metadata['combined_data'] = combined_wfm_data
    print("Saving cache…")
    with open(pickle_file, 'wb') as pf:
        pickle.dump(wfm_stream_metadata, pf)

print("combined_data shape:", combined_wfm_data.shape)
waveform_times_synchronized = False

Loading cached array…
combined_data shape: (160873947, 2)
